In [3]:
!pwd

/storage/users/job37yv/Projects/DECIDE_A04_CdiffOrganoid_Butyrate/notebooks


In [4]:
import pandas as pd, re, os
fc = pd.read_csv('../results/tables/counts_raw.tsv', sep='\t', comment='#')
fc.index = [g.split('.')[0] for g in fc['Geneid']]
cts = fc.iloc[:, 6:]; cts.columns = [re.sub(r'_Aligned.*','',os.path.basename(c)) for c in cts.columns]
for sym,gid in [('AR','ENSG00000169083'),('TMPRSS2','ENSG00000184012'),('BHLHA15','ENSG00000180535')]:
    if gid in cts.index:
        r = cts.loc[gid]
        print(f"{sym} {gid}: total={int(r.sum())} max={int(r.max())} n>=10={(r>=10).sum()}/18")
    else: print(f"{sym}: not in matrix")

AR ENSG00000169083: total=18 max=5 n>=10=0/18
TMPRSS2 ENSG00000184012: total=162682 max=15250 n>=10=18/18
BHLHA15 ENSG00000180535: total=2832 max=435 n>=10=18/18


In [5]:
# 2. Confirm the GTF gene_id we queried actually maps to AR (rule out ID drift)
import subprocess
print(subprocess.run(
  "grep -P 'gene_name \"AR\"\\b' data/refs/human_gencode/gencode.v46.primary_assembly.annotation.gtf | head -1",
  shell=True, capture_output=True, text=True).stdout)

In [7]:
# A06 expressed AR? read its count table directly
import pandas as pd, glob
f = '../../epithelial_coinfection-model_WORKING/data/processed_data/qc_fractional_counts/counts_fractional.tsv'
a06 = pd.read_csv(f, sep='\t', index_col=0)
ar = [i for i in a06.index if 'ENSG00000169083' in str(i) or i=='AR']
print(a06.loc[ar]) if ar else print('AR not in A06 matrix either — check how A06 derived the AR candidacy')

AR not in A06 matrix either — check how A06 derived the AR candidacy


In [8]:
import pandas as pd, numpy as np, os, re
from scipy.stats import spearmanr   # falls back below if missing

# A) Did A06 actually EXPRESS AR, or was it a network-inferred node?
A06='../../epithelial_coinfection-model_WORKING'; AR='ENSG00000169083'
for f in [f'{A06}/data/processed_data/qc_fractional_counts/counts_fractional.tsv',
          f'{A06}/analysis/expr_mean/expr_mean_novers.csv',
          f'{A06}/analysis/integration/master_gene_table.csv']:
    if not os.path.exists(f): print('skip(missing):',f); continue
    df=pd.read_csv(f, sep=None, engine='python', index_col=0)
    idx=[i for i in df.index if AR in str(i) or str(i)=='AR']
    print('\n==',os.path.basename(f),'=='); print(df.loc[idx].to_string() if idx else 'AR not found; cols='+str(list(df.columns)[:6]))

# B) A04: do chaperone and chemokine modules co-vary per-sample (coupling), beyond the infection split?
fc=pd.read_csv('../results/tables/counts_raw.tsv',sep='\t',comment='#'); fc.index=[g.split('.')[0] for g in fc['Geneid']]
cts=fc.iloc[:,6:]; cts.columns=[re.sub(r'_Aligned.*','',os.path.basename(c)) for c in cts.columns]
ln=np.log2(cts/cts.sum()*1e6+1)
ids={'HSPA5':'ENSG00000044574','CALR':'ENSG00000179218','WFS1':'ENSG00000109501','HYOU1':'ENSG00000149428',
     'CXCL1':'ENSG00000163739','CXCL2':'ENSG00000081041','CXCL3':'ENSG00000163734','CXCL8':'ENSG00000169429'}
E=pd.DataFrame({s:ln.loc[g] for s,g in ids.items() if g in ln.index}).T
samp=E.columns; inf=pd.Series(['Cd' if s.startswith('Cd') else 'Mock' for s in samp],index=samp)
chap=E.loc[['HSPA5','CALR','WFS1','HYOU1']].mean(); chemo=E.loc[['CXCL1','CXCL2','CXCL3','CXCL8']].mean()
def rho(a,b):
    try: return spearmanr(a,b)
    except NameError: return (pd.Series(a).corr(pd.Series(b),method='spearman'), np.nan)
for lab,m in [('all 18',pd.Series(True,index=samp)),('within Cd',inf=='Cd'),('within Mock',inf=='Mock')]:
    r,p=rho(chap[m],chemo[m]); print(f'{lab:11s} n={int(m.sum()):2d}  chaperone~chemokine rho={r:+.2f} p={p:.3g}')
r,p=rho(E.loc['HSPA5'],E.loc['CXCL8']); print(f'\nHSPA5~CXCL8 all18 rho={r:+.2f} p={p:.3g}')
m=inf=='Cd'; r,p=rho(E.loc['HSPA5'][m],E.loc['CXCL8'][m]); print(f'HSPA5~CXCL8 Cd    rho={r:+.2f} p={p:.3g}')


== counts_fractional.tsv ==
AR not found; cols=['19543', '19697', '20644', '20645', '20646', '20647']

== expr_mean_novers.csv ==
                           Bt         Cd     Co       Mock UniProtID gene_symbol
ENSG00000169083.18  74.533333  77.383333  97.96  69.793333    P10275          AR

== master_gene_table.csv ==
                    logFC_Bt_vs_Mock  padj_Bt_vs_Mock  logFC_Cd_vs_Mock  padj_Cd_vs_Mock  logFC_Co_vs_Mock  padj_Co_vs_Mock  logFC_Bt_vs_Cd  padj_Bt_vs_Cd  logFC_Bt_vs_Co  padj_Bt_vs_Co  logFC_Cd_vs_Co  padj_Cd_vs_Co pathway
ENSG00000169083.18          0.269576         0.904711          0.324177         0.999665           0.85181         0.063573       -0.053062       0.979779       -0.578903       0.434775        -0.53771       0.180707   Other
all 18      n=18  chaperone~chemokine rho=+0.87 p=2.71e-06
within Cd   n= 9  chaperone~chemokine rho=+0.70 p=0.0358
within Mock n= 9  chaperone~chemokine rho=+0.50 p=0.17

HSPA5~CXCL8 all18 rho=+0.74 p=0.000447
HSPA5~CXCL8 Cd   

In [1]:
#!/usr/bin/env python3
# =============================================================================
# tf_activity_A04.py — who controls the coupled UPR/chemokine module?
# TF-activity inference with decoupleR + CollecTRI on A04 contrasts.
# Nominates TFs whose activity FALLS as butyrate rises in Cd (candidate shared
# controllers of both the chaperone and chemokine arms), restricted to genes
# actually expressed in A04 (no topology-only nodes -> the A06 AR trap avoided).
#
# Run in an env with: decoupler, pandas, numpy  (pip install decoupler omnipath)
#   A04_PROJ=~/Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate python scripts/tf_activity_A04.py
# Uses the DE_*.csv stat (Wald 'stat' or signed -log10p) as the gene-level input.
# =============================================================================
import os, sys, re
from pathlib import Path
import numpy as np, pandas as pd

PROJ = Path(os.environ.get("A04_PROJ", Path.cwd()))
TAB  = PROJ/"results/tables"; FIG = PROJ/"results/figures"
OUT  = PROJ/"results/tf_activity"; OUT.mkdir(parents=True, exist_ok=True)
GTF  = PROJ/"data/refs/human_gencode/gencode.v46.primary_assembly.annotation.gtf"

# ---- map Ensembl -> symbol (decoupler/CollecTRI work on symbols) ------------
def gtf_symbol_map(gtf):
    d={}; gid=re.compile(r'gene_id "([^".]+)'); gnm=re.compile(r'gene_name "([^"]+)"')
    if not Path(gtf).exists(): return d
    with open(gtf) as fh:
        for ln in fh:
            if ln.startswith("#"): continue
            f=ln.split("\t")
            if len(f)>2 and f[2]=="gene":
                a,b=gid.search(ln),gnm.search(ln)
                if a and b: d[a.group(1)]=b.group(1)
    return d
sym = gtf_symbol_map(GTF)

try:
    import decoupler as dc
except ImportError:
    sys.exit("decoupler not installed -> pip install decoupler omnipath")

# ---- CollecTRI regulons (expressed-gene restricted later) -------------------
print(">> fetching CollecTRI...")
net = dc.get_collectri(organism="human", split_complexes=False)

contrasts = ["Butyrate_10v0_Cd","Butyrate_10v0_Mock",
             "Cd_vs_Mock_0mM","Cd_vs_Mock_10mM"]
def load_stat(tag):
    f = TAB/f"DE_{tag}.csv"
    if not f.exists(): print(f"   skip (missing) {f}"); return None
    df = pd.read_csv(f, index_col=0)
    # gene-level signed statistic: prefer Wald 'stat', else signed -log10(padj)
    if "stat" in df.columns: s = df["stat"]
    else:
        s = np.sign(df["log2FoldChange"]) * -np.log10(df["padj"].clip(lower=1e-300))
    s.index = [sym.get(re.sub(r"\..*","",g), g) for g in s.index]
    s = s[~s.index.duplicated()].dropna()
    return s

mats = {t: load_stat(t) for t in contrasts}
mats = {t:s for t,s in mats.items() if s is not None}
mat = pd.DataFrame(mats).T.fillna(0.0)          # contrasts x genes
print(f">> stat matrix: {mat.shape[0]} contrasts x {mat.shape[1]} genes")

# restrict regulons to genes present (expressed) in A04 -> no phantom TFs
net_expr = net[net["target"].isin(mat.columns)]
print(f">> CollecTRI edges kept (expressed targets): {len(net_expr)} / {len(net)}")

# ---- ULM activity -----------------------------------------------------------
acts, pvals = dc.run_ulm(mat=mat, net=net_expr, source="source", target="target",
                         weight="weight", min_n=5, verbose=True)
acts.to_csv(OUT/"tf_activity_scores.csv")
pvals.to_csv(OUT/"tf_activity_pvals.csv")

# ---- the question: which TFs LOSE activity as butyrate rises in Cd? ---------
key = "Butyrate_10v0_Cd"
if key in acts.index:
    a = acts.loc[key].sort_values()
    p = pvals.loc[key]
    res = pd.DataFrame({"activity": a, "pval": p.reindex(a.index)})
    res["fdr"] = (res["pval"]*len(res)).clip(upper=1)   # rough BH-ish
    res.to_csv(OUT/"TF_butyrate_in_Cd_ranked.csv")
    print("\n#### TFs DOWN with butyrate in Cd (candidate module controllers) ####")
    print(res.head(15).round(3).to_string())
    print("\n#### TFs UP with butyrate in Cd ####")
    print(res.tail(8).round(3).to_string())
    # sanity: do the usual inflammatory TFs (NFKB1/RELA) drop? does any chaperone TF drop?
    for tf in ["NFKB1","RELA","RELB","SP1","SREBF2","XBP1","ATF4","ATF6","CREB3L2","NFE2L2","NFYA","CEBPB"]:
        if tf in acts.columns:
            print(f"   {tf:8s} activity@but10v0_Cd = {acts.loc[key,tf]:+.2f}  (p={pvals.loc[key,tf]:.2g})")

# ---- cross-contrast heatmap of top movers ----------------------------------
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    top = acts.loc[key].abs().sort_values(ascending=False).head(25).index if key in acts.index else acts.columns[:25]
    H = acts[top].T
    fig,ax=plt.subplots(figsize=(7, .3*len(top)+1))
    im=ax.imshow(H.values, cmap="RdBu_r", vmin=-4, vmax=4, aspect="auto")
    ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top, fontsize=7)
    fig.colorbar(im, label="TF activity (ULM)"); ax.set_title("TF activity across A04 contrasts")
    plt.tight_layout(); fig.savefig(FIG/"tf_activity_heatmap.png", dpi=130)
    print(f"\n>> wrote {FIG/'tf_activity_heatmap.png'} and {OUT}/TF_butyrate_in_Cd_ranked.csv")
except Exception as e:
    print("plot skipped:", e)
print("\nDONE. The TFs at the TOP of the 'down with butyrate in Cd' list that ALSO")
print("rise with infection (Cd_vs_Mock_0mM) are the candidate shared controllers.")

/home/job37yv/miniforge3/envs/decoupler/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>> fetching CollecTRI...


AttributeError: module 'decoupler' has no attribute 'get_collectri'

In [2]:
# ---- CollecTRI regulons (version-robust) ------------------------------------
print(">> fetching CollecTRI...")
try:                                   # decoupler >= 2.0
    net = dc.op.collectri(organism="human")
except AttributeError:
    try:                               # some 2.x builds
        net = dc.dt.collectri(organism="human")
    except AttributeError:             # decoupler 1.x
        net = dc.get_collectri(organism="human", split_complexes=False)
print(f">> CollecTRI: {len(net)} edges, cols={list(net.columns)}")

>> fetching CollecTRI...
>> CollecTRI: 42990 edges, cols=['source', 'target', 'weight', 'resources', 'references', 'sign_decision']


In [3]:
# ---- ULM activity (version-robust) ------------------------------------------
# decoupler 2.x: dc.mt.ulm returns an AnnData-like / writes .obsm; 1.x: dc.run_ulm
try:                                   # 1.x signature
    acts, pvals = dc.run_ulm(mat=mat, net=net_expr, source="source",
                             target="target", weight="weight", min_n=5, verbose=True)
except AttributeError:                 # 2.x signature
    import anndata as ad
    ad_mat = ad.AnnData(mat)
    dc.mt.ulm(data=ad_mat, net=net_expr, tmin=5)
    acts  = ad_mat.obsm["score_ulm"]
    pvals = ad_mat.obsm["padj_ulm"] if "padj_ulm" in ad_mat.obsm else ad_mat.obsm["pvals_ulm"]

NameError: name 'mat' is not defined